# Term Project: Reinterpreting mESC LIN28A Data via Immunological Filter

이 노트북은 `plan.md`에서 정의된 **[Phase 1]** 과 **[Phase 2]** 파트를 수행합니다.

- **Phase 1**: BAM 파일 탐색 및 GTF 주석을 기반으로 한 Read Counting (`featureCounts` 활용), 데이터 프레임 병합 및 노이즈 필터링
- **Phase 2**: RPF와 RNA-seq 간의 번역 효율(TE) 산출 및 LIN28A 결핍 시의 $\Delta TE$ (`log2_fold_change`) 계산

In [1]:
import os
import glob
import pandas as pd
import numpy as np
import subprocess

# 1. BAM 및 GTF 파일 탐색
bam_files = glob.glob('./**/binfo1-datapack1/**/*.bam', recursive=True)
if not bam_files:
    bam_files = glob.glob('./**/*.bam', recursive=True)
    
gtf_files = glob.glob('./**/*.gtf', recursive=True)
gtf_file = gtf_files[0] if gtf_files else 'gencode.gtf'

print("=== Found BAM files ===")
for f in bam_files:
    print(f)
print("\n=== Found GTF file ===")
print(gtf_file)


=== Found BAM files ===
./binfo1-datapack1/CLIP-let7g.bam
./binfo1-datapack1/RNA-siLin28a.bam
./binfo1-datapack1/RNA-control.bam
./binfo1-datapack1/RNA-siLuc.bam
./binfo1-datapack1/CLIP-35L33G.bam
./binfo1-datapack1/RPF-siLin28a.bam
./binfo1-datapack1/RPF-siLuc.bam

=== Found GTF file ===
./gencode.gtf


In [2]:
# 2. 조건별 BAM 파일 식별 (이름에 siLuc, siLin28a, RPF, RNA 포함 여부로 매핑)
target_bams = {'siLuc_RPF': '', 'siLin28a_RPF': '', 'siLuc_RNAseq': '', 'siLin28a_RNAseq': ''}
for f in bam_files:
    fname = os.path.basename(f).lower()
    if 'siluc' in fname and 'rpf' in fname: target_bams['siLuc_RPF'] = f
    elif 'silin28a' in fname and 'rpf' in fname: target_bams['siLin28a_RPF'] = f
    elif 'siluc' in fname and 'rna' in fname: target_bams['siLuc_RNAseq'] = f
    elif 'silin28a' in fname and 'rna' in fname: target_bams['siLin28a_RNAseq'] = f

print("=== Target BAM Mapping ===")
for k, v in target_bams.items():
    print(f"{k}: {v}")

# featureCounts 실행 명령어 구성
bam_list_str = ' '.join([v for v in target_bams.values() if v])
output_counts = 'gene_counts.txt'

if bam_list_str:
    cmd = f"featureCounts -T 4 -a {gtf_file} -o {output_counts} {bam_list_str}"
    print("\nExecuting featureCounts... (이 작업은 리드 수에 따라 시간이 다소 걸릴 수 있습니다)")
    print(cmd)
    
    # 쉘 명령어 실행
    !{cmd}
else:
    print("Target BAM files are not fully identified. Please check file names.")

=== Target BAM Mapping ===
siLuc_RPF: ./binfo1-datapack1/RPF-siLuc.bam
siLin28a_RPF: ./binfo1-datapack1/RPF-siLin28a.bam
siLuc_RNAseq: ./binfo1-datapack1/RNA-siLuc.bam
siLin28a_RNAseq: ./binfo1-datapack1/RNA-siLin28a.bam

Executing featureCounts... (이 작업은 리드 수에 따라 시간이 다소 걸릴 수 있습니다)
featureCounts -T 4 -a ./gencode.gtf -o gene_counts.txt ./binfo1-datapack1/RPF-siLuc.bam ./binfo1-datapack1/RPF-siLin28a.bam ./binfo1-datapack1/RNA-siLuc.bam ./binfo1-datapack1/RNA-siLin28a.bam

        ==========     _____ _    _ ____  _____  ______          _____  
        =====         / ____| |  | |  _ \|  __ \|  ____|   /\   |  __ \ 
          =====      | (___ | |  | | |_) | |__) | |__     /  \  | |  | |
            ====      \___ \| |  | |  _ <|  _  /|  __|   / /\ \ | |  | |
              ====    ____) | |__| | |_) | | \ \| |____ / ____ \| |__| |
        ==========   |_____/ \____/|____/|_|  \_\______/_/    \_\_____/
	  v2.1.1

//========================== featureCounts setting ========================

In [3]:
# 3. DataFrame 로드 및 전처리 (Phase 1 결론)
if os.path.exists(output_counts):
    raw_counts = pd.read_csv(output_counts, sep='\t', comment='#')
    
    rename_dict = {}
    for col in raw_counts.columns:
        for cond, bam_path in target_bams.items():
            if bam_path and bam_path in col:
                rename_dict[col] = cond
    
    df_counts = raw_counts.rename(columns=rename_dict)
    cols_to_keep = ['Geneid', 'Length'] + list(target_bams.keys())
    cols_to_keep = [c for c in cols_to_keep if c in df_counts.columns]
    df = df_counts[cols_to_keep].copy()
    
    # Low-count expression 필터링 (로우 카운트 합 > 10)
    sample_cols = [c for c in target_bams.keys() if c in df.columns]
    df['total_counts'] = df[sample_cols].sum(axis=1)
    df_filtered = df[df['total_counts'] > 10].copy()
    df_filtered.drop(columns=['total_counts'], inplace=True)
    
    print(f"Original genes: {len(df)}, Filtered genes (counts > 10): {len(df_filtered)}")
    display(df_filtered.head())


Original genes: 55359, Filtered genes (counts > 10): 20075


,Geneid,Length,siLuc_RPF,siLin28a_RPF,siLuc_RNAseq,siLin28a_RNAseq
10,ENSMUSG00000103161.2,3012,0,0,13,9
15,ENSMUSG00000102343.2,1364,2,1,157,207
19,ENSMUSG00000025902.14,4772,3,2,5,8
21,ENSMUSG00000102269.2,2991,9,2,11,28
26,ENSMUSG00000098104.2,1470,7,2,15,26


In [4]:
# 4. Translation Efficiency (TE) & Delta TE 계산 (Phase 2)
if 'df_filtered' in locals():
    # RPM 계산용 라이브러리 사이즈 계산
    lib_sizes = df_filtered[sample_cols].sum()
    pc = 1 # pseudocount (분모 0 및 log(0) 방지)
    
    for col in sample_cols:
        rpm_col = col + '_RPM'
        df_filtered[rpm_col] = (df_filtered[col] + pc) * 1e6 / lib_sizes[col]
        
    # TE = RPF RPM / RNA-seq RPM (유전자 길이가 상쇄되므로 RPM 기반 비율 계산 가능)
    if 'siLuc_RPF_RPM' in df_filtered.columns and 'siLuc_RNAseq_RPM' in df_filtered.columns:
        df_filtered['TE_siLuc'] = df_filtered['siLuc_RPF_RPM'] / df_filtered['siLuc_RNAseq_RPM']
        
    if 'siLin28a_RPF_RPM' in df_filtered.columns and 'siLin28a_RNAseq_RPM' in df_filtered.columns:
        df_filtered['TE_siLin28a'] = df_filtered['siLin28a_RPF_RPM'] / df_filtered['siLin28a_RNAseq_RPM']
        
    # Delta TE = log2(TE_siLin28a / TE_siLuc)
    if 'TE_siLin28a' in df_filtered.columns and 'TE_siLuc' in df_filtered.columns:
        df_filtered['log2_fold_change'] = np.log2(df_filtered['TE_siLin28a'] / df_filtered['TE_siLuc'])
        
    print("=== Phase 2 Calculation Complete ===")
    display(df_filtered[['Geneid', 'TE_siLuc', 'TE_siLin28a', 'log2_fold_change']].head())


=== Phase 2 Calculation Complete ===


,Geneid,TE_siLuc,TE_siLin28a,log2_fold_change
10,ENSMUSG00000103161.2,0.053800,0.115262,1.099235
15,ENSMUSG00000102343.2,0.014301,0.011083,-0.367814
19,ENSMUSG00000025902.14,0.502134,0.384206,-0.386192
21,ENSMUSG00000102269.2,0.627668,0.119236,-2.396176
26,ENSMUSG00000098104.2,0.376601,0.128069,-1.556117
